In [ ]:
import yaml
from itertools import product

def load_architecture_profiles(yaml_path):
    """Return sorted list of architecture_profile names from architectures.yaml."""
    with open(yaml_path, 'r') as f:
        archs = yaml.safe_load(f)
    return sorted(list(archs.keys()))

def generate_test_sweep(
    architectures_yaml,
    lag_windows,
    epochs,
    learning_rates,
    data_sample,
    batch_sizes,
    physics_strategies,
    seed,
    output_file
):
    arch_profiles = load_architecture_profiles(architectures_yaml)
    experiments = []

    for (arch, lag, ep, lr, data_sample, batch, phys, seed) in product(
        arch_profiles, lag_windows, epochs, learning_rates, data_sample, batch_sizes, physics_strategies, seed
    ):
        exp = {
            "experiment_id": f"{arch}_lag{lag}_ep{ep}_lr{lr}_batch{batch}_{phys}",
            "lag_window": lag,
            "epochs": ep,
            "learning_rate": lr,
            "data_sample": data_sample,
            "batch_size": batch,
            "physics_strategy": phys,
            "architecture_profile": arch,
            "seed": seed
        }
        experiments.append(exp)

    with open(output_file, 'w') as f:
        yaml.dump(experiments, f, sort_keys=False, default_flow_style=False, allow_unicode=True)

    print(f"Generated {len(experiments)} experiments in {output_file}")

# Example usage:
if __name__ == "__main__":
    generate_test_sweep(
        architectures_yaml="architectures_PINNs.yaml",
        lag_windows=[7, 15, 30],
        epochs=[10, 50, 100, 200],
        learning_rates=[1e-2, 1e-3, 1e-4],
        data_sample = [0.25, 0.5, 0.9],
        batch_sizes=[16, 32],
        physics_strategies=["pressure_ensemble", "arps", "combined_exp_arps", "exponential", "static"],
        seed=[42],
        output_file="test_sweep.yml"
    )

In [ ]:
from hpo.search_space import define_search_space, define_fast_search_space
import optuna

params_dict = {
    "architecture_profile": "arch_identity_01_bias_scale",  # escolha um dos nomes da sua lista
    "lag_window": 7,
    "epochs": 10,
    "learning_rate": 1e-3,
    "batch_size": 16,
    "physics_strategy": "pressure_ensemble",
    "data_sample": 0.25
}

trial = optuna.trial.FixedTrial(params_dict)  # Or use Optuna's suggest API in an actual study
params = define_fast_search_space(trial)
print(params)

In [ ]:
# --- Test Notebook for the Profile Manager ---
# This notebook verifies that the `profile_manager` module can correctly:
# 1. Load a profile from a YAML file.
# 2. Expand it with configurations from `architectures_PINNs.yaml`.
# 3. Validate the final configuration against the schema.
# 4. Generate a unique ID if one is not provided.

import sys
import os
import json
from pprint import pprint

# Add the project's 'src' directory to the Python path
# This allows us to import our module directly
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
src_path = os.path.join(project_root, 'src')
if src_path not in sys.path:
    sys.path.insert(0, src_path)

# Import the function we want to test
from profile_manager import load_and_expand_profile

print("Setup complete. Test environment is ready.")